In [3]:
import pandas as pd

In [4]:
df = pd.read_csv("../data/raw/hannousse_2021/dataset_B_05_2020.csv")
features = df.columns.drop(["url", "status"])

In [5]:
n0 = len(df)
df = df.drop_duplicates(subset="url", keep="first")       # même URL collectée 2 fois
df = df.drop_duplicates(subset=features, keep="first")    # mêmes caractéristiques (campagnes)
df = df.reset_index(drop=True)

print(n0, "->", len(df), f"({n0 - len(df)} lignes retirées)")
print(df["status"].value_counts())

11430 -> 11255 (175 lignes retirées)
status
legitimate    5715
phishing      5540
Name: count, dtype: int64


In [6]:
import tldextract

In [7]:
extract_priv = tldextract.TLDExtract(include_psl_private_domains=True)

df["d_icann"] = df["url"].apply(lambda u: tldextract.extract(u).top_domain_under_public_suffix)
df["d_priv"] = df["url"].apply(lambda u: extract_priv(u).top_domain_under_public_suffix)

for col in ["d_icann", "d_priv"]:
    print(col, ":", df[col].nunique(), "groupes")
    print(df[col].value_counts().head(8), "\n")

d_icann : 6566 groupes
d_icann
duilawyeryork.com    331
blogspot.com         324
appspot.com          257
wikipedia.org        177
google.com           149
                      94
tumblr.com            88
sharepoint.com        74
Name: count, dtype: int64 

d_priv : 7125 groupes
d_priv
duilawyeryork.com    331
wikipedia.org        177
google.com           149
                      94
tumblr.com            88
sharepoint.com        74
wordpress.com         63
000webhostapp.com     60
Name: count, dtype: int64 



In [8]:
from urllib.parse import urlparse

In [9]:
# Plateformes où chaque sous-domaine est un site distinct, absentes de la liste des suffixes privés
HOSTING = {"sharepoint.com", "000webhostapp.com", "tumblr.com", "wordpress.com"}

def group_key(url):
    domain = extract_priv(url).top_domain_under_public_suffix
    host = urlparse(url).hostname or ""
    if domain == "" or domain in HOSTING:   # IP ou plateforme d'hébergement -> nom d'hôte
        return host
    return domain

df["group"] = df["url"].apply(group_key)
print(df["group"].nunique(), "groupes")
print((df["group"] == "").sum(), "groupes vides")
print(df["group"].value_counts().head(8))


7438 groupes
0 groupes vides
group
duilawyeryork.com          331
wikipedia.org              177
google.com                 149
shadetreetechnology.com     53
academic.ru                 50
webcindario.com             49
thefreedictionary.com       48
codesandbox.io              45
Name: count, dtype: int64


In [10]:
from sklearn.model_selection import StratifiedGroupKFold

In [11]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(sgkf.split(df, df["status"], groups=df["group"]))

train = df.iloc[train_idx].reset_index(drop=True)
test = df.iloc[test_idx].reset_index(drop=True)

print("train :", len(train), "| test :", len(test), f"({len(test) / len(df):.1%})")
print(train["status"].value_counts(normalize=True).round(3))
print(test["status"].value_counts(normalize=True).round(3))
print("groupes communs train/test :", len(set(train["group"]) & set(test["group"])))


train : 9003 | test : 2252 (20.0%)
status
legitimate    0.508
phishing      0.492
Name: proportion, dtype: float64
status
legitimate    0.508
phishing      0.492
Name: proportion, dtype: float64
groupes communs train/test : 0


In [12]:
import hashlib
from pathlib import Path

In [13]:
out = Path("../data/processed")
out.mkdir(parents=True, exist_ok=True)

train.to_csv(out / "train.csv", index=False)
test.to_csv(out / "test_LOCKED.csv", index=False)

sha = hashlib.sha256((out / "test_LOCKED.csv").read_bytes()).hexdigest()
print("SHA-256 du test :", sha)


SHA-256 du test : 8cf8d62e96bd9dd6a37accdbfb65c26316f5e7689ef7511fa96f1718c6059be1
